# ML pipeline: predicting NLRB union election outcomes

In this notebook we train and tune several models on `data/elections_clean.csv` and look at how their test scores hold up across several random states.

Sections:

0. Imports
1. Load the data and pick the features
2. Group-based test split
3. Preprocessor
4. Evaluation metric and baseline
5. Pipeline functions: `ML_pipeline_groups_GridSearchCV` and `ML_pipeline_groups_xgb`
6. Models: each model gets its own cell with the classifier, its `param_grid`, and whether it needs imputation and the final scaler
7. Model comparison

To try a different metric, change `scoring` in section 4. 
To change a model or its parameters, edit that model's cell in section 6.

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
import joblib
import xgboost
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import get_scorer
from sklearn.metrics import log_loss
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import warnings
warnings.filterwarnings("ignore")   # as in L19: hides the IterativeImputer convergence warnings

## 1. Load the data and pick the features

`elections_clean.csv` is made by `src/preprocessmanual.py`. It has one row per single-union RC election (15,296 rows). 
We already dropped everything that's only known after the vote in that script (vote counts, closing date, closing reason).

### Feature types:

- **Numerical** (`num_ftrs`), scaled with StandardScaler:
    - `eligible`: the size of the bargaining unit
    - `days_petition_to_election`: how long it took from the petition to the vote
    - `election_year`: the year of the vote. It's technically ordinal, but it's already a number with even spacing between years, so we just scale it like a continuous feature. Want to impute missing year, rather than treat it as earliest/latest category.
    - `employer_prior_wins` and `employer_prior_losses`: how many elections the union won and lost at the same employer on an *earlier* date. These come from public NLRB records, so they'd be known before the vote. They let the model pick up on an employer's track record without using the employer's name/id as a feature. If it's an employer's first election, both are 0.

- **Categorical** (`cat_ftrs`), one-hot encoded:
    - `parent_union` 
    - `state`
    - `employer_has_lawyer` 
    - `has_intervenor` 

### Columns we don't use as features:

- **group ID**:
    - `employer` is the group ID we split on. If we used it as a feature too, the model could just memorize employers.

- **ID**
    - `case_number`, `case_name`, `city` and `union_name` are IDs or free text.
    
- **Leakage**
    - `fiscal_year` is the year of the report where the case was listed as **closed**. That can be a lot later than the vote, often because of objections filed after the election, so using it would leak post-vote information. We use `election_year` instead.



In [ ]:
df = pd.read_csv('../data/elections_clean.csv', parse_dates=['election_date'])

label = 'union_won'          # 1 = union won (Certification of Representative), 0 = lost
y = df[label]
groups = df['employer']      # the group ID (L11)

num_ftrs = ['eligible', 'days_petition_to_election', 'election_year',
            'employer_prior_wins', 'employer_prior_losses']
cat_ftrs = ['parent_union', 'state', 'employer_has_lawyer', 'has_intervenor']
# sensitivity run only (may include lawyers who joined after the vote):
# num_ftrs = num_ftrs + ['n_participants', 'n_employer_lawyers', 'n_petitioner_lawyers']
# cat_ftrs = cat_ftrs + ['petitioner_has_lawyer']

X = df[num_ftrs + cat_ftrs]

print(X.shape)
print('fraction of elections the union won:', np.around(y.mean(), 3))
print('missing values per feature:')
print(X.isnull().sum())

## 2. Group-based test split

About half the elections come from employers that show up more than once (Starbucks alone has 728). If the same employer ended up in both the training and test data, the model could score well just by recognizing the employer, and the test score would look better than it really is. So we split by employer:
every case belongs to one employer, so all elections from a case automatically end up together.

- **Test set:** `GroupShuffleSplit` holds out 20% of the *employers*.
- **CV on `_other`:** `GroupKFold` keeps every employer inside a single fold.
- **No stratification:** the union won about 71% of elections, which isn't extremely imbalanced

In [ ]:
def train_test_split_groups(X, y, groups, random_state):
    # create a test set based on groups (same code as in L11's ML_pipeline_groups_GridSearchCV)
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=random_state)
    for i_other, i_test in splitter.split(X, y, groups):
        X_other, y_other, groups_other = X.iloc[i_other], y.iloc[i_other], groups.iloc[i_other]
        X_test, y_test, groups_test = X.iloc[i_test], y.iloc[i_test], groups.iloc[i_test]
    # check the split: no employer is in both sets
    assert len(set(groups_other) & set(groups_test)) == 0
    return X_other, X_test, y_other, y_test, groups_other


# check the splits for the random states we will use
nr_states = 5
for i in range(nr_states):
    X_other, X_test, y_other, y_test, groups_other = train_test_split_groups(X, y, groups, 42*i)
    print('random state', 42*i, '| other:', len(y_other), 'test:', len(y_test),
          '| win rate other:', np.around(y_other.mean(), 3), 'test:', np.around(y_test.mean(), 3),
          '| starbucks in test:', 'starbucks' in set(groups.loc[X_test.index]))

## 3. Preprocessor

### Preprocessors:
- **Numerical features:** StandardScaler. Set to ignore NaN
- **Categorical features:** missing values are first filled with the string `'missing'` so they become their own category, then everything is one-hot encoded. unknown category gets all-zero columns.

### Handling Missing values

- Missing values are all continuous variables
    - `eligible` (157 rows) (numerical)
    - `days_petition_to_election` (736) (numerical)
    - `election_year` (26) (numerical)
    - `employer_prior_wins`, `employer_prior_losses` (the same 26 elections that have no election date) (numerical)

- Logistic regression and KNN use `IterativeImputer` after the preprocessor. It predicts each missing value from all the other features (multivariate imputation).
- XGBoost and sklearn's random forest can handle NaN on their own, so their pipeline is just `preprocessor` followed by the model.

### Final scaler
- After the preprocessor, only the numerical features have mean 0 and std 1
- the one-hot columns are still 0/1. So we add `final_scaler = StandardScaler()` right before the model to put every column on the same scale, allowing to compare logistic regression coefficients as feature importances

In [ ]:
def make_preprocessor():
    # standard scaler (keeps NaN in place)
    numeric_transformer = Pipeline(steps=[
        ('scaler', StandardScaler())])
    # one-hot encoder
    # We need to replace the NaN with a string first!
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))])
    # collect the transformers
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, num_ftrs),
            ('cat', categorical_transformer, cat_ftrs)])
    return preprocessor


def make_imputer(random_state):
    # L18: IterativeImputer with a random forest as the estimator (L18 used n_estimators=1 for speed)
    return IterativeImputer(estimator=RandomForestRegressor(n_estimators=1, random_state=random_state),
                            skip_complete=True, random_state=random_state)


# look at the preprocessed training set of one split
X_other, X_test, y_other, y_test, groups_other = train_test_split_groups(X, y, groups, 0)
preprocessor = make_preprocessor()
X_prep = preprocessor.fit_transform(X_other)
feature_names = preprocessor.get_feature_names_out()
df_train = pd.DataFrame(data=X_prep, columns=feature_names)
print(df_train.shape)
print(feature_names)
print('missing values left after the preprocessor:', df_train.isnull().sum().sum())

## 4. Evaluation metric and baseline

To try a different metric, change `scoring`.
### Metric

**log loss** 
- In sklearn the scorer is `'neg_log_loss'`. Meaning max is better (closer to 0)

- **Pros:**
    - Union organizers would make decisions based on the predicted *probability* of winning. If chances are low, then they need to improve the campaign. 
    - We want to reduce FN. FN means you are told you are likely to win when you are not. Then the campaign wouldn't be improved, and it would be a waste of resources/expenses.
    - FP means that the organizers are told they are likely to lose when they are not. They are unlikely to drop the entire campaign. Otherwise they improve an already prepared campaign.
    - Thus, we need the probabilities to be reliable. Log loss scores the predicted probabilities directly and penalizes confident mistakes the most.

- **Cons:** 
    - it treats both kinds of mistakes the same. But missing a loss is worse, because a lost election blocks a new one in that unit for 12 months (NLRA Section 9(c)(3)). 
    - We deal with that later when picking the probability cutoff for the confusion matrix (using F2, more weight to recall, with "union loses" as the positive class).

### Baseline 
Out baseline is a model that just predicts the training win rate for every election. Our models need to beat its log loss on each test set.

In [ ]:
scoring = 'neg_log_loss'

def baseline_score(y_other, y_test):
    # predict the win rate of the training data for every test election
    y_test_pred = np.full((len(y_test), 2), [1 - y_other.mean(), y_other.mean()])
    return -log_loss(y_test, y_test_pred)   # negated, like 'neg_log_loss'

baseline_scores = np.zeros(nr_states)
for i in range(nr_states):
    X_other, X_test, y_other, y_test, groups_other = train_test_split_groups(X, y, groups, 42*i)
    baseline_scores[i] = baseline_score(y_other, y_test)
print('baseline test score:', np.around(np.mean(baseline_scores), 4), '+/-', np.around(np.std(baseline_scores), 4))

## 5. Pipeline functions

### GridSearchCV 
`ML_pipeline_groups_GridSearchCV` takes the model and its `param_grid` as inputs, so we can plug in any model:

- **`clf`:** the classifier. We set its `random_state` to the current run's random state
- **`param_grid`:** the parameters to tune. The names start with the model's name in lowercase, e.g. `'logisticregression__C'`.
- **`impute`:** 
    - `True` adds the IterativeImputer and the final scaler (logistic regression, KNN). 
    - `False` leaves NaN in for the model to handle (random forest)

It returns the fitted `grid`, `X_test` and `y_test`

### XGBoost with Early Stopping
`ML_pipeline_groups_xgb` 

- it loops over the `ParameterGrid`
- it uses `XGB.set_params(**params, early_stopping_rounds=50)` with `eval_set` set to the validation set
- Uses the same 5 GroupKFold folds, so XGBoost is tuned on the same folds as the other models
- the final model is trained on all of `_other` using the best parameters and the average best number of trees across the folds, so we don't have to hold back any data for early stopping

In [ ]:
def ML_pipeline_groups_GridSearchCV(X, y, groups, random_state, n_folds, clf, param_grid, impute):
    # create a test set based on groups
    X_other, X_test, y_other, y_test, groups_other = train_test_split_groups(X, y, groups, random_state)
    # splitter for _other
    kf = GroupKFold(n_splits=n_folds)
    # the classifier with this random state
    clf = clone(clf)
    if 'random_state' in clf.get_params():
        clf.set_params(random_state=random_state)
    # create the pipeline: preprocessor (+ imputer + final scaler) + supervised ML method
    preprocessor = make_preprocessor()
    if impute:
        final_scaler = StandardScaler()
        pipe = make_pipeline(preprocessor, make_imputer(random_state), final_scaler, clf)
    else:
        pipe = make_pipeline(preprocessor, clf)
    # prepare gridsearch
    grid = GridSearchCV(pipe, param_grid=param_grid, scoring=scoring,
                        cv=kf, return_train_score=True, n_jobs=-1)
    # do kfold CV on _other
    grid.fit(X_other, y_other, groups=groups_other)
    return grid, X_test, y_test


def ML_pipeline_groups_xgb(X, y, groups, random_state, n_folds, param_grid):
    # create a test set based on groups
    X_other, X_test, y_other, y_test, groups_other = train_test_split_groups(X, y, groups, random_state)
    # splitter for _other
    kf = GroupKFold(n_splits=n_folds)
    scorer = get_scorer(scoring)

    pg = ParameterGrid(param_grid)
    val_scores = np.zeros((len(pg), n_folds))
    best_iterations = np.zeros((len(pg), n_folds))

    for f, (train_index, val_index) in enumerate(kf.split(X_other, y_other, groups_other)):
        X_train, y_train = X_other.iloc[train_index], y_other.iloc[train_index]
        X_val, y_val = X_other.iloc[val_index], y_other.iloc[val_index]
        # fit the preprocessor on the training fold only
        preprocessor = make_preprocessor()
        df_train = preprocessor.fit_transform(X_train)
        df_val = preprocessor.transform(X_val)
        for p in range(len(pg)):
            params = pg[p]
            XGB = xgboost.XGBClassifier(random_state=random_state, eval_metric='logloss')
            XGB.set_params(**params, early_stopping_rounds=50)
            XGB.fit(df_train, y_train, eval_set=[(df_val, y_val)], verbose=False)   # with early stopping
            val_scores[p, f] = scorer(XGB, df_val, y_val)
            best_iterations[p, f] = XGB.best_iteration

    # the best parameter set has the highest mean validation score
    p_best = np.argmax(val_scores.mean(axis=1))
    best_params = pg[p_best]
    n_trees = int(best_iterations[p_best].mean()) + 1

    # train the final model on _other with the best parameters and the average best number of trees
    XGB = xgboost.XGBClassifier(random_state=random_state, eval_metric='logloss')
    XGB.set_params(**best_params)
    XGB.set_params(n_estimators=n_trees)
    pipe = make_pipeline(make_preprocessor(), XGB)
    pipe.fit(X_other, y_other)

    cv_results = pd.DataFrame({'params': list(pg),
                               'mean_test_score': val_scores.mean(axis=1),
                               'std_test_score': val_scores.std(axis=1),
                               'n_trees': best_iterations.mean(axis=1).astype(int) + 1})
    return pipe, cv_results, best_params, n_trees, X_test, y_test


def run_model(name, clf, param_grid, impute, n_folds=5):
    # loop through nr_states random states: nr_states best models and nr_states test scores (L9, L11)
    test_scores = np.zeros(nr_states)
    cv_scores = np.zeros(nr_states)
    final_models = []
    best_params = []
    for i in range(nr_states):
        random_state = 42*i
        print('random state', random_state)
        if name == 'xgb':
            pipe, cv_results, params, n_trees, X_test, y_test = ML_pipeline_groups_xgb(
                X, y, groups, random_state, n_folds, param_grid)
            params = {**params, 'n_estimators': n_trees}
            cv_scores[i] = cv_results['mean_test_score'].max()
            final_models.append(pipe)
        else:
            grid, X_test, y_test = ML_pipeline_groups_GridSearchCV(
                X, y, groups, random_state, n_folds, clf, param_grid, impute)
            params = grid.best_params_
            cv_scores[i] = grid.best_score_    # the mean validation score of the best parameters
            final_models.append(grid.best_estimator_)
        best_params.append(params)
        test_scores[i] = get_scorer(scoring)(final_models[-1], X_test, y_test)
        print('   best model parameters:', params)
        print('   validation score:', np.around(cv_scores[i], 4))
        print('   test score:', np.around(test_scores[i], 4), '| baseline:', np.around(baseline_scores[i], 4))
    print(name, 'test score:', np.around(np.mean(test_scores), 4), '+/-', np.around(np.std(test_scores), 4))

    # save the results; X_test and y_test can be recreated with train_test_split_groups(X, y, groups, 42*i)
    joblib.dump({'name': name, 'scoring': scoring, 'num_ftrs': num_ftrs, 'cat_ftrs': cat_ftrs,
                 'random_states': [42*i for i in range(nr_states)],
                 'final_models': final_models, 'best_params': best_params,
                 'cv_scores': cv_scores, 'test_scores': test_scores, 'baseline_scores': baseline_scores},
                f'../results/{name}.joblib')
    return test_scores, final_models

## 6. Models

Run the cell for the model that will be used.

### Parameter grids:

- **Logistic regression, Elastic Net:** 
    - `C` on a log scale
    - `l1_ratio` set to 0 (L2 penalty) or 1 (L1 penalty) 
- **KNN:** 
    - number of neighbors and the weighting. 
- **Random forest:** 
    - `max_depth` 
    - `max_features`.
- **XGBoost:** 
    - `learning_rate`
    - `n_estimators`
    - `missing`
    - `colsample_bytree` 
    - `subsample`
    - `max_depth`
    - `reg_alpha` 
    - `reg_lambda`.

In [ ]:
# logistic regression (L8, L9, L20)
param_grid = {'logisticregression__C': np.logspace(-4, 4, 9),
              'logisticregression__l1_ratio': [0, 1]}
clf = LogisticRegression(solver='saga', max_iter=10000)
test_scores_lr, final_models_lr = run_model('logreg', clf, param_grid, impute=True)

In [ ]:
# KNN (L14)
param_grid = {'kneighborsclassifier__n_neighbors': [1, 3, 10, 30, 100, 300],
              'kneighborsclassifier__weights': ['uniform', 'distance']}
clf = KNeighborsClassifier()
test_scores_knn, final_models_knn = run_model('knn', clf, param_grid, impute=True)

In [ ]:
# random forest (L15); sklearn's random forest handles NaN itself, so no imputer
param_grid = {'randomforestclassifier__max_depth': [1, 3, 5, 10, 30, None],
              'randomforestclassifier__max_features': [0.1, 0.3, 0.5, 0.7, 0.9]}
clf = RandomForestClassifier(n_estimators=300)
test_scores_rf, final_models_rf = run_model('rf', clf, param_grid, impute=False)

In [ ]:
# XGBoost (L19); handles NaN itself, so no imputer
param_grid = {"learning_rate": [0.03],
              "n_estimators": [10000],
              "missing": [np.nan],
              "max_depth": [1, 3, 5, 10],
              "reg_alpha": [0e0, 1e-1, 1e0, 1e1],
              "reg_lambda": [1e-1, 1e0, 1e1],
              "colsample_bytree": [0.9],
              "subsample": [0.66]}
test_scores_xgb, final_models_xgb = run_model('xgb', None, param_grid, impute=False)

In [ ]:
# SVC (L11, L17): slow, run last if at all
param_grid = {'svc__C': np.logspace(-2, 3, 6),
              'svc__gamma': np.logspace(-3, 1, 5)}
clf = SVC(probability=True)   # probability=True is needed for log loss
# test_scores_svc, final_models_svc = run_model('svc', clf, param_grid, impute=True)

## 7. Model comparison

This loads the saved results for every model in `results/`

For each model we plot the mean validation score with error bars (the std across random states) and compare it to the baseline.

In [ ]:
names = []
cv_mean, cv_std, test_mean, test_std = [], [], [], []
for name in ['logreg', 'knn', 'rf', 'xgb', 'svc']:
    try:
        results = joblib.load(f'../results/{name}.joblib')
    except FileNotFoundError:
        continue
    if results['scoring'] != scoring:
        continue
    names.append(name)
    cv_mean.append(np.mean(results['cv_scores']))
    cv_std.append(np.std(results['cv_scores']))
    test_mean.append(np.mean(results['test_scores']))
    test_std.append(np.std(results['test_scores']))

summary = pd.DataFrame({'model': names, 'cv_mean': cv_mean, 'cv_std': cv_std,
                        'test_mean': test_mean, 'test_std': test_std})
print(summary.round(4))
print('baseline test score:', np.around(np.mean(baseline_scores), 4), '+/-', np.around(np.std(baseline_scores), 4))

plt.figure(figsize=(6, 4))
plt.errorbar(range(len(names)), cv_mean, yerr=cv_std, fmt='o', capsize=4, label='models')
plt.axhline(np.mean(baseline_scores), linestyle='--', color='gray', label='baseline')
plt.xticks(range(len(names)), names)
plt.ylabel('mean CV score (' + scoring + ')')
plt.legend()
plt.tight_layout()
plt.savefig('../figures/cv_scores_by_model.png', dpi=300)
plt.show()